# Création de la base d'apprentissage

## 1. Présentation générale

Ce notebook constitue l'étape cruciale de notre projet : la **réconciliation de données multi-sources** pour bâtir notre dataset d'entraînement. L'objectif est d'aligner les statistiques de performance des joueurs avec leurs valeurs marchandes respectives.

Nous centralisons alors dans un premier temps des fichiers issus de l'API Soccerdata, regroupant ainsi des données FBref et des données Understat. De plus, nous centralisons également des données Transfermarkt (les données financières ainsi que notre variable cible : la valeur marchande) et du mapping issu de worldfootballR.

Nous réalisons ensuite une fusion à plusieurs niveaux : nous utilisons les identifiants connus des joueurs puisdu fuzzy-mapping.


Nous devrions obtenir finalement une table prête pour de plus profondes analyses voire pour de la modélisation, mêlant ainsi des données issues des performances sportives à des données analysant la valeur marchande des joueurs de football.

Pour ce faire, nous importons dans un premier temps des packages et des fonctions nécessaires à la création de notre base d'apprentissage.

In [53]:
# Importation des packages nécessaires

import pandas as pd
import os
import sys

# On connecte le notebook à tous les fichiers inclus dans le dossier /fonctions
sys.path.append(os.path.abspath("../fonctions"))

%load_ext autoreload
%autoreload 2

from merging import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 2. Chargement et préparation des sources


Nous importons dans un premier temps nos trois fichiers comprenant nos données :
- issues du mapping de worldfootballR
- issues de transfermarkt
- issues de soccerdata

In [54]:
# Chargement des données du mapping
df_mapping_initial = pd.read_csv("../data_finale/mapping_worldfootballR/mapping_fbref_tm.csv", encoding='latin1')

# Chargement des données du dataset Soccerdata
df_soccerdata_initial = pd.read_csv("../data/soccerdata/data_final_soccerdata.csv")

# Chargement des données du dataset Transfermarkt
df_players = pd.read_csv("../data/transfermarkt_datasets/players.csv")
df_valuations = pd.read_csv("../data/transfermarkt_datasets/player_valuations.csv")

# Préparation des données de Transfermarkt
df_tm_initial = prepare_transfermarkt_data(
    df_players,
    df_valuations
)

# Chargement des données du dataset de blessures Transfermarkt
df_blessures = pd.read_csv("../data/dataset_blessures.csv")

# Préparation des données de blessures Transfermarkt
df_blessures_initial = aggregate_injuries_by_season(df_blessures)

Plutôt que de traiter chaque dataframe manuellement ici, nous utilisons la fonction match_player_data. Cette fonction encapsule toute la logique de nettoyage définie précédemment :

- Correction de l'encoding : Application de fix_encoding sur les noms FBref.

- Normalisation des noms : Suppression des accents, mise en minuscule et nettoyage des caractères spéciaux via normalize_name.

- Harmonisation des dates : Extraction de l'année de naissance (dob_key) pour faciliter le matching entre les sources.

- Création de clés composites : Génération de clés basées sur "Prénom + Nom" pour Transfermarkt.

In [55]:
# Nous appliquons les logiques décrites ci-dessus
df_mapping, df_soccerdata, df_tm, df_blessures = match_player_data(df_mapping_initial, df_soccerdata_initial,
                                                      df_tm_initial, df_blessures_initial)

In [56]:
df_tm

,player_id,valuation_season_year,first_name,last_name,name,last_season,current_club_id,player_code,country_of_birth,city_of_birth,...,url,current_club_domestic_competition_id,current_club_name,highest_market_value_in_eur,date,market_value_in_eur,player_club_domestic_competition_id,join_key,join_key_full,dob_key
0,3333,2019.0,James,Milner,James Milner,2025,1237,james-milner,England,Leeds,...,https://www.transfermarkt.co.uk/james-milner/p...,GB1,Brighton and Hove Albion Football Club,21000000.0,2020-04-08,6500000.0,GB1,james milner,james milner,1986
1,3333,2020.0,James,Milner,James Milner,2025,1237,james-milner,England,Leeds,...,https://www.transfermarkt.co.uk/james-milner/p...,GB1,Brighton and Hove Albion Football Club,21000000.0,2021-06-08,3000000.0,GB1,james milner,james milner,1986
2,3333,2021.0,James,Milner,James Milner,2025,1237,james-milner,England,Leeds,...,https://www.transfermarkt.co.uk/james-milner/p...,GB1,Brighton and Hove Albion Football Club,21000000.0,2022-06-15,2000000.0,GB1,james milner,james milner,1986
3,3333,2022.0,James,Milner,James Milner,2025,1237,james-milner,England,Leeds,...,https://www.transfermarkt.co.uk/james-milner/p...,GB1,Brighton and Hove Albion Football Club,21000000.0,2023-06-20,1500000.0,GB1,james milner,james milner,1986
4,3333,2023.0,James,Milner,James Milner,2025,1237,james-milner,England,Leeds,...,https://www.transfermarkt.co.uk/james-milner/p...,GB1,Brighton and Hove Albion Football Club,21000000.0,2024-05-27,1000000.0,GB1,james milner,james milner,1986
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19225,1375227,2025.0,Thiago,Pitarch,Thiago Pitarch,2025,418,thiago-pitarch,Spain,Fuenlabrada,...,https://www.transfermarkt.co.uk/thiago-pitarch...,ES1,Real Madrid Club de Fútbol,20000000.0,2026-03-16,20000000.0,ES1,thiago pitarch,thiago pitarch,2007
19226,1380977,2025.0,Samba,Konaté,Samba Konaté,2025,23826,samba-konate,France,Le Havre,...,https://www.transfermarkt.co.uk/samba-konate/p...,L1,RasenBallsport Leipzig,1000000.0,2025-12-06,1000000.0,NaN,samba konate,samba konate,2009
19227,1390649,2024.0,Yan,Diomande,Yan Diomande,2025,23826,yan-diomande,Cote d'Ivoire,Abidjan,...,https://www.transfermarkt.co.uk/yan-diomande/p...,L1,RasenBallsport Leipzig,90000000.0,2025-06-09,1500000.0,ES1,yan diomande,yan diomande,2006
19228,1390649,2025.0,Yan,Diomande,Yan Diomande,2025,23826,yan-diomande,Cote d'Ivoire,Abidjan,...,https://www.transfermarkt.co.uk/yan-diomande/p...,L1,RasenBallsport Leipzig,90000000.0,2026-03-20,75000000.0,L1,yan diomande,yan diomande,2006


## 3. La fusion à multi-niveaux

Nous appliquons ensuite une stratégie de fusion des bases de données en 2 étapes pour maximiser le taux de correspondance.

Dans un premier temps, nous réalisons une jointure exacte via le dictionnaire de mapping.

Ensuite, nous effectuons une recherche plus floue (fuzzy) sur le mapping avec un seuil supérieur à 90%.

In [57]:
df_final, still_missing = run_player_matching(df_soccerdata, df_mapping, df_tm, df_blessures)

[1] Nom exact (mapping)     : 16280 | restants : 833
[2] Fuzzy nom (mapping)     :   283 | restants : 550
[3.1] Match direct TM (Exact) :   239 | restants : 311
[3.2] Match direct TM (Fuzzy) :    56 | restants : 248


Nous pouvons enfin importer notre base d'apprentissage sous le format CSV.

In [58]:
df_final.to_csv(r'..\data_finale\base_apprentissage.csv', index=False, sep=',', encoding='utf-8-sig')
still_missing.to_csv(r'..\data_finale\analyse_orphelins\still_missing.csv', index=False, sep=',', encoding='utf-8-sig')
df_soccerdata.to_csv(r'..\data_finale\analyse_orphelins\soccerdata.csv', index=False, sep=',', encoding='utf-8-sig')

In [ ]:
df_final

,league,season,team,player,nation,pos,age,born,Playing Time_MP,Playing Time_Starts,...,injury_minor_unknown_nb_d,injury_minor_unknown_nb_m,injury_musculaire,injury_genou,injury_cheville_pied,injury_mollet_tibia,injury_dos_bassin,injury_trauma_severe,injury_medical_repos,injury_minor_unknown
0,ENG-Premier League,2021,Arsenal,Ainsley Maitland-Niles,ENG,"MF,DF",22,1997.0,11,5,...,12.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,ENG-Premier League,2021,Arsenal,Alexandre Lacazette,FRA,FW,29,1991.0,31,22,...,23.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,ENG-Premier League,2021,Arsenal,Bernd Leno,GER,GK,28,1992.0,35,35,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,ENG-Premier League,2021,Arsenal,Bukayo Saka,ENG,MF,18,2001.0,32,30,...,4.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
4,ENG-Premier League,2021,Arsenal,Calum Chambers,ENG,DF,25,1995.0,10,8,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16853,ITA-Serie A,2526,Lecce,Nikola Štulić,SRB,FW,24-238,2001.0,32,20,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
16854,ITA-Serie A,2526,Parma,Ben Cremaschi,USA,MF,21-063,2005.0,8,2,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
16855,ITA-Serie A,2526,Pisa,Filip Stojilković,SUI,"FW,MF",26-120,2000.0,10,5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
16856,ITA-Serie A,2526,Pisa,İsak Vural,TUR,MF,19-341,2006.0,12,7,...,40.0,3.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0


## **Les joueurs orphelins**

In [60]:
still_missing = pd.read_csv(r'..\data_finale\analyse_orphelins\still_missing.csv', encoding='utf-8-sig')
df_soccerdata = pd.read_csv(r'..\data_finale\analyse_orphelins\soccerdata.csv', encoding='utf-8-sig')

In [61]:
nb_joueurs_orphelins = still_missing['join_key'].nunique()
nb_joueurs_total = df_soccerdata['join_key'].nunique()

print(f"Joueurs orphelins : {nb_joueurs_orphelins}")
print(f"Taux joueurs orphelins : {nb_joueurs_orphelins / nb_joueurs_total:.2%}")

Joueurs orphelins : 237
Taux joueurs orphelins : 3.83%


In [62]:
orphelins_par_saison = (
    still_missing
    .groupby('season_year')
    .size()
    .sort_index()
)

print(orphelins_par_saison)

season_year
2020      1
2021     16
2022     25
2023     41
2024     12
2025    153
dtype: int64


In [63]:
temp_data = still_missing[still_missing['season_year']<2025]
temp_data

,league,season,team,player,nation,pos,age,born,Playing Time_MP,Playing Time_Starts,...,Performance_PKcon,Performance_OG,xg,xa,np_xg,xg_chain,xg_buildup,join_key,dob_key,season_year
0,ENG-Premier League,2021,Manchester Utd,Will Fish,ENG,DF,17,2003.0,1,0,...,NaN,0,NaN,NaN,NaN,NaN,NaN,will fish,2003,2020
1,ENG-Premier League,2223,Brighton,Cameron Peupion,AUS,MF,19,2002.0,1,0,...,NaN,0,0.053613,0.000000,0.053613,0.000000,0.000000,cameron peupion,2002,2022
2,ENG-Premier League,2223,Manchester City,Shea Charles,NIR,"DF,MF",18,2003.0,1,0,...,NaN,0,0.000000,0.000000,0.000000,0.129427,0.129427,shea charles,2003,2022
3,ENG-Premier League,2223,Southampton,Kami Doyle,ENG,MF,16,2005.0,1,0,...,NaN,0,NaN,NaN,NaN,NaN,NaN,kami doyle,2005,2022
4,ENG-Premier League,2223,Tottenham Hotspur,George Abbott,ENG,MF,16,2005.0,1,0,...,NaN,0,NaN,NaN,NaN,NaN,NaN,george abbott,2005,2022
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
221,ITA-Serie A,2324,Salernitana,Gerardo Fusco,ITA,FW,18,2005.0,1,0,...,NaN,0,0.000000,0.029909,0.000000,0.000000,0.000000,gerardo fusco,2005,2023
222,ITA-Serie A,2324,Salernitana,Mateusz Łęgowski,POL,MF,20,2003.0,29,10,...,NaN,0,NaN,NaN,NaN,NaN,NaN,mateusz u0141 u0119gowski,2003,2023
223,ITA-Serie A,2324,Udinese,Antonio Tikvić,CRO,DF,19,2004.0,1,0,...,NaN,0,NaN,NaN,NaN,NaN,NaN,antonio tikvi u0107,2004,2023
224,ITA-Serie A,2425,Lecce,Filip Marchwiński,POL,MF,22,2002.0,1,0,...,NaN,0,NaN,NaN,NaN,NaN,NaN,filip marchwi u0144ski,2002,2024


In [64]:
df_soccerdata[df_soccerdata["player"] == "Cameron Peupion"]

,league,season,team,player,nation,pos,age,born,Playing Time_MP,Playing Time_Starts,...,Performance_PKcon,Performance_OG,xg,xa,np_xg,xg_chain,xg_buildup,join_key,dob_key,season_year
1191,ENG-Premier League,2223,Brighton,Cameron Peupion,AUS,MF,19,2002.0,1,0,...,NaN,0,0.053613,0.0,0.053613,0.0,0.0,cameron peupion,2002.0,2022


In [65]:
df_mapping[df_mapping["PlayerFBref"].str.contains("Peupion")]

,PlayerFBref,fbref_id,tm_id,TmPos,join_key


In [66]:
df_tm[df_tm["last_name"] == "Peupion"]

,player_id,valuation_season_year,first_name,last_name,name,last_season,current_club_id,player_code,country_of_birth,city_of_birth,...,url,current_club_domestic_competition_id,current_club_name,highest_market_value_in_eur,date,market_value_in_eur,player_club_domestic_competition_id,join_key,join_key_full,dob_key


In [67]:
from fuzzywuzzy import fuzz, process
import pandas as pd

# Initialisation des listes pour séparer les index
mapping_keys = df_mapping['join_key'].dropna().tolist()
indices_caches = []
indices_vrais_absents = []

print("Analyse des lignes en cours...")

# Classement ligne par ligne de still_missing
for idx, row in still_missing.iterrows():
    player_clean = str(row['player']).lower().strip()
    
    # Si le joueur est déjà un match exact parfait (normalement 0 ici)
    if player_clean in mapping_keys:
        indices_caches.append(idx)
        continue
        
    # Test fuzzy de contrôle (score à 75)
    best_match = process.extractOne(player_clean, mapping_keys, scorer=fuzz.token_set_ratio)
    
    if best_match and best_match[1] >= 75:
        indices_caches.append(idx)
    else:
        indices_vrais_absents.append(idx)

# Création des deux DataFrames cibles
df_caches = still_missing.loc[indices_caches].copy()
df_vrais_absents = still_missing.loc[indices_vrais_absents].copy()

# Colonnes d'affichage standard pour l'inspection
cols_affichage = ['player', 'team', 'season', 'league']


print(f"\nCas 1 : cachés par l'orthographe")
# On trie par équipe et joueur pour que ce soit lisible
print(df_caches[cols_affichage].sort_values(by=['team', 'player']).head(30).to_string(index=False))


print(f"\nCas 2 : vrais absents")
print(df_vrais_absents[cols_affichage].sort_values(by=['team', 'player']).head(30).to_string(index=False))

Analyse des lignes en cours...

Cas 1 : cachés par l'orthographe
             player            team  season             league
    Lander Pinillos          Alavés    2526        ESP-La Liga
         Marc Tenas          Alavés    2122        ESP-La Liga
   Youssef Lekhedim          Alavés    2526        ESP-La Liga
    Marciano Tchami         Almería    2324        ESP-La Liga
         Oumar Pona          Angers    2526        FRA-Ligue 1
    George Hemmings     Aston Villa    2526 ENG-Premier League
       Asier Hierro   Athletic Club    2526        ESP-La Liga
      Antonio Gomis Atlético Madrid    2223        ESP-La Liga
       Javier Boñar Atlético Madrid    2526        ESP-La Liga
     Yusuf Kabadayı        Augsburg    2425     GER-Bundesliga
  Mamoudou Cissokho         Auxerre    2526        FRA-Ligue 1
     Jofre Torrents       Barcelona    2526        ESP-La Liga
      Tomas Marques       Barcelona    2526        ESP-La Liga
        Bara Ndiaye   Bayern Munich    2526     GER-B

In [68]:
df_caches

,league,season,team,player,nation,pos,age,born,Playing Time_MP,Playing Time_Starts,...,Performance_PKcon,Performance_OG,xg,xa,np_xg,xg_chain,xg_buildup,join_key,dob_key,season_year
1,ENG-Premier League,2223,Brighton,Cameron Peupion,AUS,MF,19,2002.0,1,0,...,NaN,0,0.053613,0.000000,0.053613,0.000000,0.000000,cameron peupion,2002,2022
2,ENG-Premier League,2223,Manchester City,Shea Charles,NIR,"DF,MF",18,2003.0,1,0,...,NaN,0,0.000000,0.000000,0.000000,0.129427,0.129427,shea charles,2003,2022
4,ENG-Premier League,2223,Tottenham Hotspur,George Abbott,ENG,MF,16,2005.0,1,0,...,NaN,0,NaN,NaN,NaN,NaN,NaN,george abbott,2005,2022
6,ENG-Premier League,2324,Bournemouth,Dominic Sadi,ENG,MF,19,2003.0,1,0,...,NaN,0,0.000000,0.000000,0.000000,0.000000,0.000000,dominic sadi,2003,2023
9,ENG-Premier League,2324,Everton,Lewis Warrington,ENG,MF,20,2002.0,1,0,...,NaN,0,0.000000,0.000000,0.000000,0.000000,0.000000,lewis warrington,2002,2023
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
239,ITA-Serie A,2526,Parma,Daniel Mikołajewski,POL,FW,20-100,2006.0,1,0,...,NaN,0,NaN,NaN,NaN,NaN,NaN,daniel miko u0142ajewski,2006,2025
240,ITA-Serie A,2526,Parma,Oliver Jensen,DEN,MF,24-055,2002.0,32,19,...,NaN,0,NaN,NaN,NaN,NaN,NaN,oliver jensen,2002,2025
242,ITA-Serie A,2526,Pisa,Louis Buffon,CZE,FW,18-127,2007.0,4,0,...,NaN,0,0.000000,0.000000,0.000000,0.028428,0.028428,louis buffon,2007,2025
244,ITA-Serie A,2526,Roma,Alessandro Romano,SUI,MF,19-321,2006.0,2,0,...,NaN,0,0.000000,0.000000,0.000000,0.533233,0.533233,alessandro romano,2006,2025


In [69]:
df_vrais_absents

,league,season,team,player,nation,pos,age,born,Playing Time_MP,Playing Time_Starts,...,Performance_PKcon,Performance_OG,xg,xa,np_xg,xg_chain,xg_buildup,join_key,dob_key,season_year
0,ENG-Premier League,2021,Manchester Utd,Will Fish,ENG,DF,17,2003.0,1,0,...,NaN,0,NaN,NaN,NaN,NaN,NaN,will fish,2003,2020
3,ENG-Premier League,2223,Southampton,Kami Doyle,ENG,MF,16,2005.0,1,0,...,NaN,0,NaN,NaN,NaN,NaN,NaN,kami doyle,2005,2022
5,ENG-Premier League,2324,Aston Villa,Finley Munroe,ENG,DF,18,2005.0,1,0,...,NaN,0,0.000000,0.000000,0.000000,0.000000,0.000000,finley munroe,2005,2023
7,ENG-Premier League,2324,Brighton,Mark O'Mahony,IRL,FW,18,2005.0,3,1,...,NaN,0,NaN,NaN,NaN,NaN,NaN,mark o mahony,2005,2023
8,ENG-Premier League,2324,Chelsea,Jimi Tauriainen,ENG,FW,19,2004.0,1,0,...,NaN,0,0.000000,0.000000,0.000000,0.000000,0.000000,jimi tauriainen,2004,2023
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
238,ITA-Serie A,2526,Milan,Cheveyo Muy,NED,FW,19-137,2006.0,2,0,...,NaN,0,NaN,NaN,NaN,NaN,NaN,cheveyo muy,2006,2025
241,ITA-Serie A,2526,Parma,Tjaš Begić,SVN,MF,22-308,2003.0,1,0,...,NaN,0,NaN,NaN,NaN,NaN,NaN,tja u0161 begi u0107,2003,2025
243,ITA-Serie A,2526,Pisa,Rafiu Durosinmi,NGA,FW,23-123,2003.0,12,4,...,NaN,0,0.759808,0.087608,0.759808,0.926448,0.182712,rafiu durosinmi,2003,2025
245,ITA-Serie A,2526,Sassuolo,Laurs Skjellerup,DEN,FW,23-265,2002.0,1,0,...,NaN,0,0.015538,0.000000,0.015538,0.015538,0.000000,laurs skjellerup,2002,2025
